# Hotel Booking Cancellation Prediction — Random Forest Classifier

**Algorithm:** Random Forest  
**Dataset:** Hotel Booking Demand Dataset  
**Objective:** Predict whether a hotel booking will be canceled (`is_canceled`)  

This notebook applies the **Random Forest** classification algorithm to predict hotel booking cancellations. The workflow covers:
1. Dataset description and exploration
2. Exploratory Data Analysis (EDA)
3. Data preprocessing and cleaning
4. Model training and evaluation
5. Results analysis and discussion

## 1. Dataset Description

**Source:** [Hotel Booking Demand Dataset — Kaggle](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
**Original Paper:** Nuno Antonio, Ana de Almeida, and Luis Nunes (2019). "Hotel booking demand datasets." *Data in Brief*, Volume 22.

### Context
This dataset contains **119,390 hotel bookings** from two types of hotels — a **Resort Hotel** and a **City Hotel** — with arrivals between July 2015 and August 2017. It includes bookings that were fulfilled and bookings that were canceled.

### Target Variable
- `is_canceled` — Binary (1 = canceled, 0 = not canceled)

### Key Attributes (32 columns)
| Attribute | Description |
|-----------|-------------|
| `hotel` | Type of hotel (Resort Hotel / City Hotel) |
| `lead_time` | Number of days between booking and arrival |
| `arrival_date_year/month/week_number/day_of_month` | Arrival date details |
| `stays_in_weekend_nights` | Weekend nights stayed/booked |
| `stays_in_week_nights` | Week nights stayed/booked |
| `adults`, `children`, `babies` | Number of guests |
| `meal` | Type of meal booked |
| `country` | Country of origin (ISO 3155-3:2013) |
| `market_segment` | Market segment (e.g., Online TA, Offline TA/TO) |
| `distribution_channel` | Booking distribution channel |
| `is_repeated_guest` | Whether the guest is a repeat guest (1/0) |
| `previous_cancellations` | Number of previous cancellations |
| `previous_bookings_not_canceled` | Number of previous non-canceled bookings |
| `reserved_room_type` | Room type reserved |
| `assigned_room_type` | Room type assigned |
| `booking_changes` | Number of changes made to the booking |
| `deposit_type` | Type of deposit made |
| `agent` | ID of the travel agent |
| `company` | ID of the company making the booking |
| `days_in_waiting_list` | Days the booking was on the waiting list |
| `customer_type` | Type of customer |
| `adr` | Average Daily Rate (price) |
| `required_car_parking_spaces` | Parking spaces required |
| `total_of_special_requests` | Number of special requests |
| `reservation_status` | Last reservation status |
| `reservation_status_date` | Date of last status update |

## 2. Import Libraries

In [2]:
# Core data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing, model, and evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import LabelEncoder

# Set plot style for cleaner visuals
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


## 3. Load and Inspect the Dataset

In [3]:
# Load the hotel bookings dataset from CSV file
df = pd.read_csv("dataset/hotel_bookings.csv")

# Display the first 5 rows to get an initial look at the data
print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Dataset Shape: 119390 rows × 32 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
# Check data types and non-null counts for each column
# This helps identify columns with missing values and incorrect dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [5]:
# Statistical summary of numerical columns
# Useful for spotting outliers and understanding value ranges
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [6]:
# Check the distribution of the target variable (is_canceled)
# Important to understand if we have a class imbalance problem
print("Target Variable Distribution:")
print(df['is_canceled'].value_counts())
print(f"\nCancellation Rate: {df['is_canceled'].mean()*100:.1f}%")

Target Variable Distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64

Cancellation Rate: 37.0%


## 4. Exploratory Data Analysis (EDA)

Before cleaning the data, we perform EDA to understand patterns, distributions, and relationships in the dataset. This helps justify our preprocessing decisions.

### 4.1 Missing Values Analysis